# Satellite

In [1]:
import argparse
from xuance.common import get_configs
from xuance.environment import make_envs
import numpy as np
import importlib
import env.sattelite_env as sat_env
from xuance.environment import REGISTRY_ENV
from xuance.environment import make_envs
from xuance.torch.agents import DDPG_Agent, PPOCLIP_Agent, DreamerV3Agent
from agents.DiffPPO import DiffusionPPOAgent  # class custom bạn đã viết

configs_dict = get_configs(file_dir="configs/DDPG_satellite_env.yaml")
configs = argparse.Namespace(**configs_dict)

importlib.reload(sat_env)
REGISTRY_ENV[configs.env_name] = sat_env.SatelliteMECEnvironment
envs = make_envs(configs)
obs, info = envs.reset()
print('envs ready:', type(envs).__name__)
print('obs shape:', obs.shape if isinstance(obs, np.ndarray) else type(obs))
print('info len:', len(info) if isinstance(info, list) else type(info))
envs.close()


envs = make_envs(configs)  # Make parallel environments.
Agent = DDPG_Agent(config=configs, envs=envs)  # Create a DDPG agent from XuanCe.
Agent.train(configs.running_steps // configs.parallels)  # Train the model for numerous steps.
Agent.save_model("final_train_model.pth")  # Save the model to model_dir.
Agent.finish()  # Finish the training.

2026-02-28 13:35:00,894 - env.sattelite_env - INFO - Initializing SystemParameters with seed=1
2026-02-28 13:35:00,896 - env.sattelite_env - INFO - Initialized Satellite-MEC Environment with 50 MGUs
2026-02-28 13:35:00,897 - env.sattelite_env - INFO - Initializing SystemParameters with seed=2
2026-02-28 13:35:00,897 - env.sattelite_env - INFO - Initialized Satellite-MEC Environment with 50 MGUs
2026-02-28 13:35:00,898 - env.sattelite_env - INFO - Initializing SystemParameters with seed=3
2026-02-28 13:35:00,898 - env.sattelite_env - INFO - Initialized Satellite-MEC Environment with 50 MGUs
2026-02-28 13:35:00,899 - env.sattelite_env - INFO - Initializing SystemParameters with seed=3
2026-02-28 13:35:00,901 - env.sattelite_env - INFO - Initialized Satellite-MEC Environment with 50 MGUs
2026-02-28 13:35:00,901 - env.sattelite_env - INFO - Initializing SystemParameters with seed=4
2026-02-28 13:35:00,902 - env.sattelite_env - INFO - Initialized Satellite-MEC Environment with 50 MGUs
2026-

envs ready: DummyVecEnv
obs shape: (3, 154)
info len: 3


100%|██████████| 5000/5000 [00:46<00:00, 106.73it/s]


In [2]:

configs_dict = get_configs(file_dir="configs/PPO_satellite_env.yaml")
configs = argparse.Namespace(**configs_dict)
importlib.reload(sat_env)
REGISTRY_ENV[configs.env_name] = sat_env.SatelliteMECEnvironment
envs = make_envs(configs)

Agent = PPOCLIP_Agent(config=configs, envs=envs)  # Create a PPOCLIP agent from XuanCe.
Agent.train(configs.running_steps // configs.parallels)  # Train the model for numerous steps.
Agent.save_model("final_train_model.pth")  # Save the model to model_dir.
Agent.finish()  # Finish the training.

2026-02-28 13:35:48,012 - env.sattelite_env - INFO - Initializing SystemParameters with seed=1
2026-02-28 13:35:48,013 - env.sattelite_env - INFO - Initialized Satellite-MEC Environment with 50 MGUs
2026-02-28 13:35:48,013 - env.sattelite_env - INFO - Initializing SystemParameters with seed=2
2026-02-28 13:35:48,013 - env.sattelite_env - INFO - Initialized Satellite-MEC Environment with 50 MGUs
2026-02-28 13:35:48,014 - env.sattelite_env - INFO - Initializing SystemParameters with seed=3
2026-02-28 13:35:48,014 - env.sattelite_env - INFO - Initialized Satellite-MEC Environment with 50 MGUs
100%|██████████| 5000/5000 [00:40<00:00, 124.21it/s]


In [ ]:
if not hasattr(DreamerV3Agent, "envs"):
    DreamerV3Agent.envs = property(lambda self: self.train_envs)

configs_dict = get_configs(file_dir="configs/Dreamer_satellite_env.yaml")
configs = argparse.Namespace(**configs_dict)

importlib.reload(sat_env)
REGISTRY_ENV[configs.env_name] = sat_env.SatelliteMECEnvironment
envs = make_envs(configs)

# quick smoke run first
Agent = DreamerV3Agent(config=configs, envs=envs)
Agent.train(configs.running_steps // configs.parallels)
Agent.save_model("dreamerv3_smoke_model.pth")
Agent.finish()

2026-02-28 13:36:28,291 - env.sattelite_env - INFO - Initializing SystemParameters with seed=1
2026-02-28 13:36:28,292 - env.sattelite_env - INFO - Initialized Satellite-MEC Environment with 50 MGUs
2026-02-28 13:36:28,292 - env.sattelite_env - INFO - Initializing SystemParameters with seed=2
2026-02-28 13:36:28,293 - env.sattelite_env - INFO - Initialized Satellite-MEC Environment with 50 MGUs
  0%|          | 0/10000 [00:00<?, ?it/s]/home/n2tp/miniconda3/envs/xuance/lib/python3.10/site-packages/xuance/torch/representations/world_model.py:704: UserWarning: Use of index_put_ on expanded tensors is deprecated. Please clone() the tensor before performing this operation. This also applies to advanced indexing e.g. tensor[indices] = tensor (Triggered internally at /opt/conda/conda-bld/pytorch_1720538438429/work/aten/src/ATen/native/TensorAdvancedIndexing.cpp:716.)
  self.recurrent_state[:, reset_envs], stochastic_state = self.rssm.get_initial_states((1, len(reset_envs)))
 19%|█▉        | 1

In [ ]:

# load config DiffPPO
configs_dict = get_configs(file_dir="configs/DiffPPO_config.yaml")
configs = argparse.Namespace(**configs_dict)

# register custom env
importlib.reload(sat_env)
REGISTRY_ENV[configs.env_name] = sat_env.SatelliteMECEnvironment

# build envs
envs = make_envs(configs)

# train
Agent = DiffusionPPOAgent(config=configs, envs=envs)
Agent.train(configs.running_steps // configs.parallels)
Agent.save_model("diffppo_smoke_model.pth")
Agent.finish()
envs.close()

2026-02-28 13:19:47,766 - sattelite_env - INFO - Initializing SystemParameters with seed=1
2026-02-28 13:19:47,767 - sattelite_env - INFO - Initialized Satellite-MEC Environment with 50 MGUs
2026-02-28 13:19:47,768 - sattelite_env - INFO - Initializing SystemParameters with seed=2
2026-02-28 13:19:47,768 - sattelite_env - INFO - Initialized Satellite-MEC Environment with 50 MGUs
100%|██████████| 5/5 [00:00<00:00, 124.12it/s]


In [9]:
!tensorboard --logdir=logs/ --port=6006


/home/n2tp/miniconda3/envs/xuance/lib/python3.10/site-packages/tensorboard/default.py:30: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
TensorFlow installation not found - running with reduced feature set.

NOTE: Using experimental fast data loading logic. To disable, pass
    "--load_fast=false" and report issues on GitHub. More details:
    https://github.com/tensorflow/tensorboard/issues/4784

E0301 13:59:24.257104 133141641901888 program.py:300] TensorBoard could not bind to port 6006, it was already in use
ERROR: TensorBoard could not bind to port 6006, it was already in use


# Memory

In [2]:
import argparse
from xuance.common import get_configs
from xuance.environment import make_envs
import numpy as np
import importlib
import env.memory_env as mem_env
from xuance.environment import REGISTRY_ENV
from xuance.environment import make_envs
from xuance.torch.agents import DDPG_Agent, PPOCLIP_Agent, DreamerV3Agent
from agents.DiffPPO import DiffusionPPOAgent  # class custom bạn đã viết

configs_dict = get_configs(file_dir="configs/DDPG_memory_env.yaml")
configs = argparse.Namespace(**configs_dict)

importlib.reload(mem_env)
REGISTRY_ENV[configs.env_name] = mem_env.GAIServiceEnv_v1
envs = make_envs(configs)
obs, info = envs.reset()
print('envs ready:', type(envs).__name__)
print('obs shape:', obs.shape if isinstance(obs, np.ndarray) else type(obs))
print('info len:', len(info) if isinstance(info, list) else type(info))
envs.close()


envs = make_envs(configs)  # Make parallel environments.
Agent = DDPG_Agent(config=configs, envs=envs)  # Create a DDPG agent from XuanCe.
Agent.train(configs.running_steps // configs.parallels)  # Train the model for numerous steps.
Agent.save_model("final_train_model.pth")  # Save the model to model_dir.
Agent.finish()  # Finish the training.

envs ready: DummyVecEnv
obs shape: (3, 60)
info len: 3


100%|██████████| 5000/5000 [00:28<00:00, 176.21it/s]


In [3]:

# load config DiffPPO
configs_dict = get_configs(file_dir="configs/DiffPPO_config.yaml")
configs = argparse.Namespace(**configs_dict)

# register custom env
importlib.reload(mem_env)
REGISTRY_ENV[configs.env_name] = mem_env.GAIServiceEnv_v1

# build envs
envs = make_envs(configs)

# train
Agent = DiffusionPPOAgent(config=configs, envs=envs)
Agent.train(configs.running_steps // configs.parallels)
Agent.save_model("diffppo_smoke_model.pth")
Agent.finish()
envs.close()

100%|██████████| 7500/7500 [03:10<00:00, 39.43it/s]


# LLM

In [1]:
import argparse
from xuance.common import get_configs
from xuance.environment import make_envs
from xuance.environment import REGISTRY_ENV
from xuance.environment import make_envs
from xuance.torch.agents import DDPG_Agent, PPOCLIP_Agent, DreamerV3Agent
from agents.DiffPPO import DiffusionPPOAgent  # class custom bạn đã viết


In [6]:
from xuance.common import get_configs
from xuance.environment import REGISTRY_ENV, make_envs
import numpy as np
import importlib
import env.LLM_env as llm_env_module

configs_dict = get_configs(file_dir="configs/DiffPPO_config_LLM.yaml")
configs = argparse.Namespace(**configs_dict)

# Register custom LLM environment.
importlib.reload(llm_env_module)
REGISTRY_ENV[configs.env_name] = llm_env_module.UAVLLMOffloadingEnv

# Quick smoke check for vectorized env construction and reset.
envs = make_envs(configs)
obs, info = envs.reset()
print('envs ready:', type(envs).__name__)
print('obs shape:', obs.shape if isinstance(obs, np.ndarray) else type(obs))
print('info len:', len(info) if isinstance(info, list) else type(info))
envs.close()

# Train DiffPPO agent.
envs = make_envs(configs)
Agent = DiffusionPPOAgent(config=configs, envs=envs)
Agent.train(configs.running_steps // configs.parallels)
Agent.save_model("final_train_model.pth")
Agent.finish()
envs.close()

envs ready: DummyVecEnv
obs shape: (2, 43)
info len: 2


  0%|          | 0/7500 [00:00<?, ?it/s]

100%|██████████| 7500/7500 [02:10<00:00, 57.60it/s]


In [12]:

# Load config (reuse DDPG-style training settings).
configs_dict = get_configs(file_dir="configs/PPO_LLM.yaml")
configs = argparse.Namespace(**configs_dict)

# Register custom LLM environment.
importlib.reload(llm_env_module)
REGISTRY_ENV[configs.env_name] = llm_env_module.UAVLLMOffloadingEnv

# Quick smoke check for vectorized env construction and reset.
envs = make_envs(configs)
obs, info = envs.reset()
print('envs ready:', type(envs).__name__)
print('obs shape:', obs.shape if isinstance(obs, np.ndarray) else type(obs))
print('info len:', len(info) if isinstance(info, list) else type(info))
envs.close()

# Train PPO agent.
envs = make_envs(configs)
Agent = PPOCLIP_Agent(config=configs, envs=envs)
Agent.train(configs.running_steps // configs.parallels)
Agent.save_model("final_train_model.pth")
Agent.finish()
envs.close()

envs ready: DummyVecEnv
obs shape: (3, 43)
info len: 3


100%|██████████| 5000/5000 [00:10<00:00, 488.67it/s]


In [10]:

# Load config (reuse DDPG-style training settings).
configs_dict = get_configs(file_dir="configs/DDPG_LLM_env.yaml")
configs = argparse.Namespace(**configs_dict)

# Register custom LLM environment.
importlib.reload(llm_env_module)
REGISTRY_ENV[configs.env_name] = llm_env_module.UAVLLMOffloadingEnv

# Quick smoke check for vectorized env construction and reset.
envs = make_envs(configs)
obs, info = envs.reset()
print('envs ready:', type(envs).__name__)
print('obs shape:', obs.shape if isinstance(obs, np.ndarray) else type(obs))
print('info len:', len(info) if isinstance(info, list) else type(info))
envs.close()

# Train DDPG agent.
envs = make_envs(configs)
Agent = DDPG_Agent(config=configs, envs=envs)
Agent.train(configs.running_steps // configs.parallels)
Agent.save_model("final_train_model.pth")
Agent.finish()
envs.close()

envs ready: DummyVecEnv
obs shape: (3, 43)
info len: 3


  0%|          | 0/5000 [00:00<?, ?it/s]

100%|██████████| 5000/5000 [00:14<00:00, 345.75it/s]


In [11]:
# Reward diagnostics for UAVLLMOffloadingEnv
import argparse
import importlib
import numpy as np
from xuance.common import get_configs
from xuance.environment import REGISTRY_ENV, make_envs
import env.LLM_env as llm_env_module

configs_dict = get_configs(file_dir="configs/DDPG_LLM_env.yaml")
configs = argparse.Namespace(**configs_dict)
importlib.reload(llm_env_module)
REGISTRY_ENV[configs.env_name] = llm_env_module.UAVLLMOffloadingEnv

envs = make_envs(configs)
obs, info = envs.reset()

reward_list = []
u_term, l_term, p_term, pen_term = [], [], [], []

num_envs = int(configs.parallels)
action_dim = int(envs.action_space.shape[0])

for _ in range(30):
    action = np.random.uniform(-1.0, 1.0, size=(num_envs, action_dim)).astype(np.float32)
    obs, rewards, terminated, truncated, infos = envs.step(action)

    rewards_np = np.asarray(rewards, dtype=np.float32).reshape(-1)
    reward_list.extend(rewards_np.tolist())

    if isinstance(infos, (list, tuple)) and len(infos) > 0:
        for item in infos:
            u_term.append(float(item.get("reward_utility_term", 0.0)))
            l_term.append(float(item.get("reward_latency_term", 0.0)))
            p_term.append(float(item.get("reward_power_term", 0.0)))
            pen_term.append(float(item.get("reward_penalty_term", 0.0)))

    if np.any(np.asarray(terminated)) or np.any(np.asarray(truncated)):
        break

envs.close()

print("reward mean/min/max:", float(np.mean(reward_list)), float(np.min(reward_list)), float(np.max(reward_list)))
print("utility term mean:", float(np.mean(u_term)) if len(u_term) else 0.0)
print("latency term mean:", float(np.mean(l_term)) if len(l_term) else 0.0)
print("power term mean:", float(np.mean(p_term)) if len(p_term) else 0.0)
print("penalty term mean:", float(np.mean(pen_term)) if len(pen_term) else 0.0)

reward mean/min/max: -9990.029231770834 -10000.0 -9700.876953125
utility term mean: 0.7691913333333333
latency term mean: 1209.5055256873923
power term mean: 2.380646178325017
penalty term mean: -12065.381145511707
